# Prototype V0.2 — DataCo real-dataset audit, leakage-safe split and feature engineering

A self-contained demonstration of the V0.2 pipeline:
**discovery → loading → audit → leakage analysis → split → feature engineering → save**.
Every step calls functions under `src/`; no data is fabricated or downloaded.

> **Scope note**: the real DataCo dataset does **not** contain native cybersecurity
> attack labels. `Late_delivery_risk` is an operational delivery outcome (late vs on
> time). V0.2 therefore provides clean, leakage-safe ML features + versioned splits
> and explicitly defers attack/anomaly modelling to a later prototype.


## 0. Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # make the repo root importable
import json
import pandas as pd
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", None)
from src.config import (RAW_DATA_DIR, DATA_AUDIT_DIR, PROCESSED_DATA_DIR,
                        DATACO_MAX_ROWS, SPLIT_STRATEGY, SPLIT_RATIOS, GLOBAL_SEED)
from src.pipeline_v02 import cap_before_split
print("ready")

ready


## 1. Dataset discovery

In [2]:
from src.data.dataset_discovery import resolve_dataco_dataset
res = resolve_dataco_dataset()
print("path     :", res.path)
print("found_raw:", res.found_in_raw)
print("prov     :", res.provenance)
print("messages :", res.messages)

path     : /mnt/e/PhD-Blockchain-AI/data/raw/DataCoSupplyChainDataset.csv
found_raw: True
prov     : raw_dir
messages : ['Found unambiguous DataCo file in data/raw: /mnt/e/PhD-Blockchain-AI/data/raw/DataCoSupplyChainDataset.csv']


## 2. Loading

In [3]:
from src.pipeline_v02 import load_dataco_frame
df = load_dataco_frame(res)
print("Loaded", df.shape, "| a stable `row_id` column was added for auditing.")
df.head(3)

Loaded (180519, 54) | a stable `row_id` column was added for auditing.


,row_id,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


## 3. Dimensions

In [4]:
print(f"n_rows={df.shape[0]:,}  n_columns={df.shape[1]}")
print(f"n_orders={df['Order Id'].nunique():,}  "
      f"n_customers={df['Order Customer Id'].nunique():,}  "
      f"n_products={df['Product Card Id'].nunique():,}")

n_rows=180,519  n_columns=54
n_orders=65,752  n_customers=20,652  n_products=118


## 4. Column inspection

In [5]:
from src.data.audit import audit_dataframe
summary = audit_dataframe(df, source_name=res.path.name, output_dir=DATA_AUDIT_DIR, save_figures=True)
print("Audit persisted to", DATA_AUDIT_DIR)
print("columns:")
for i, col in enumerate(summary["column_names"]):
    print(f"  {i:>2}  {col}")

Audit persisted to /mnt/e/PhD-Blockchain-AI/results/data_audit
columns:
   0  row_id
   1  Type
   2  Days for shipping (real)
   3  Days for shipment (scheduled)
   4  Benefit per order
   5  Sales per customer
   6  Delivery Status
   7  Late_delivery_risk
   8  Category Id
   9  Category Name
  10  Customer City
  11  Customer Country
  12  Customer Email
  13  Customer Fname
  14  Customer Id
  15  Customer Lname
  16  Customer Password
  17  Customer Segment
  18  Customer State
  19  Customer Street
  20  Customer Zipcode
  21  Department Id
  22  Department Name
  23  Latitude
  24  Longitude
  25  Market
  26  Order City
  27  Order Country
  28  Order Customer Id
  29  order date (DateOrders)
  30  Order Id
  31  Order Item Cardprod Id
  32  Order Item Discount
  33  Order Item Discount Rate
  34  Order Item Id
  35  Order Item Product Price
  36  Order Item Profit Ratio
  37  Order Item Quantity
  38  Sales
  39  Order Item Total
  40  Order Profit Per Order
  41  Order Regio

## 5. Column types

In [6]:
print(df.dtypes.value_counts().to_string())
print()

str        24
int64      15
float64    15



In [7]:
role = df.dtypes.astype(str).to_frame("dtype")
role["missing"] = df.isna().sum()
role.head(15)

,dtype,missing
row_id,int64,0
Type,str,0
Days for shipping (real),int64,0
Days for shipment (scheduled),int64,0
Benefit per order,float64,0
Sales per customer,float64,0
Delivery Status,str,0
Late_delivery_risk,int64,0
Category Id,int64,0
Category Name,str,0


## 6. Missing-value analysis

In [8]:
print("overview:", summary["missing_summary"])
mv = pd.read_csv(DATA_AUDIT_DIR / "missing_values.csv")
mv[mv["missing_count"] > 0].sort_values("missing_percentage", ascending=False)[:12]

overview: {'columns_with_missing': 4, 'total_cells': 9748026, 'total_missing_cells': 336209, 'overall_missing_pct': 0.03449}


,column,missing_count,missing_percentage
47,Product Description,180519,1.000000
44,Order Zipcode,155679,0.862397
15,Customer Lname,8,0.000044
20,Customer Zipcode,3,0.000017


## 7. Duplicate analysis

In [9]:
print(json.dumps(summary["duplicates"], indent=1))
df[df.duplicated(keep=False)].head(3)

{
 "fully_duplicate_rows": 0,
 "duplicate_rows_by_key_Order_Item_Id": 0,
 "duplicate_rows_by_key_Order_Id": 114767,
 "duplicate_sample_n": 0
}


,row_id,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode


## 8. Numerical statistics

In [10]:
df[summary["numeric_columns"]].describe().T.head(22)

,count,mean,std,min,25%,50%,75%,max
row_id,180519.0,90259.000000,52111.490959,0.000000,45129.500000,90259.000000,135388.500000,180518.000000
Days for shipping (real),180519.0,3.497654,1.623722,0.000000,2.000000,3.000000,5.000000,6.000000
Days for shipment (scheduled),180519.0,2.931847,1.374449,0.000000,2.000000,4.000000,4.000000,4.000000
Benefit per order,180519.0,21.974989,104.433526,-4274.979980,7.000000,31.520000,64.800003,911.799988
Sales per customer,180519.0,183.107609,120.043670,7.490000,104.379997,163.990005,247.399994,1939.989990
Late_delivery_risk,180519.0,0.548291,0.497664,0.000000,0.000000,1.000000,1.000000,1.000000
Category Id,180519.0,31.851451,15.640064,2.000000,18.000000,29.000000,45.000000,76.000000
Customer Id,180519.0,6691.379495,4162.918106,1.000000,3258.500000,6457.000000,9779.000000,20757.000000
Customer Zipcode,180516.0,35921.126914,37542.461122,603.000000,725.000000,19380.000000,78207.000000,99205.000000
Department Id,180519.0,5.443460,1.629246,2.000000,4.000000,5.000000,7.000000,12.000000


## 9. Categorical analysis

In [11]:
cats = df[summary["categorical_columns"]].nunique().sort_values(ascending=False)
cats.to_frame("n_unique").assign(cardinality=lambda t: t["n_unique"].apply(
    lambda n: "low (one-hot)" if n <= 15 else "high (metadata only)"))

,n_unique,cardinality
order date (DateOrders),65752,high (metadata only)
shipping date (DateOrders),63701,high (metadata only)
Customer Street,7458,high (metadata only)
Order City,3597,high (metadata only)
Customer Lname,1109,high (metadata only)
Order State,1089,high (metadata only)
Customer Fname,782,high (metadata only)
Customer City,563,high (metadata only)
Order Country,164,high (metadata only)
Product Image,118,high (metadata only)


## 10. Date/time analysis

In [12]:
dates = pd.to_datetime(df["order date (DateOrders)"], errors="coerce")
print("range:", dates.min(), "->", dates.max())
print("orders/week:", round(len(dates) / (dates.max() - dates.min()).days * 7, 1))
dates.dt.year.value_counts().sort_index().to_frame("orders")

range: 2015-01-01 00:00:00 -> 2018-01-31 23:38:00
orders/week: 1122.2


,orders
order date (DateOrders),
2015,62650
2016,62550
2017,53196
2018,2123


## 11. Identifier analysis

In [13]:
for c in ["Order Id", "Order Item Id", "Order Customer Id", "Product Card Id"]:
    print(f"{c:<20} unique={df[c].nunique():>7,}  of {len(df):,} rows")
print("\nauditor role flags ->", summary["potential_identifier_columns"])

Order Id             unique= 65,752  of 180,519 rows
Order Item Id        unique=180,519  of 180,519 rows
Order Customer Id    unique= 20,652  of 180,519 rows
Product Card Id      unique=    118  of 180,519 rows

auditor role flags -> ['row_id', 'Late_delivery_risk', 'Category Id', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Latitude', 'Longitude', 'Order City', 'Order Country', 'Order Customer Id', 'Order Id', 'Order Item Cardprod Id', 'Order Item Id', 'Order State', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image']


## 12. Leakage analysis

In [14]:
print("candidate leakage fields (from audit):")
for cand in summary["leakage_candidates"]:
    print("  -", cand)
print("\nconstant columns :", summary["constant_columns"][:10])
print("near-constant    :", summary["near_constant_columns"][:10])
print("high-cardinality :", summary["high_cardinality_categorical_columns"][:10])

candidate leakage fields (from audit):
  - {'column': 'Days for shipping (real)', 'reason': 'outcome_adjacent', 'suggestion': 'exclude from ML features; traceability metadata only'}
  - {'column': 'Benefit per order', 'reason': 'aggregate', 'suggestion': 'verify aggregation window; prefer train-derived aggregates'}
  - {'column': 'Sales per customer', 'reason': 'aggregate', 'suggestion': 'verify aggregation window; prefer train-derived aggregates'}
  - {'column': 'Delivery Status', 'reason': 'outcome', 'suggestion': 'exclude from ML features; traceability metadata only'}
  - {'column': 'Late_delivery_risk', 'reason': 'target', 'suggestion': 'used as supervised target, never as a feature'}
  - {'column': 'Customer Id', 'reason': 'key_shared', 'suggestion': 'split by this key or drop from features'}
  - {'column': 'Order Customer Id', 'reason': 'key_shared', 'suggestion': 'split by this key or drop from features'}
  - {'column': 'Order Id', 'reason': 'key_shared', 'suggestion': 'split by

## 13. Feature engineering

In [15]:
from src.preprocessing.feature_engineering_v2 import (feature_catalog_df,
                                          build_raw_ml_features, compute_order_level,
                                          categorical_source_columns)
print("Feature catalogue (%d entries):" % len(feature_catalog_df()))
print(feature_catalog_df()[["name", "source", "category", "definition"]].to_string(index=False))

Feature catalogue (17 entries):
                    name                                  source    category                                                            definition
     order_item_quantity                     Order Item Quantity transaction                     Quantity of the product ordered on the item line.
order_item_product_price                Order Item Product Price transaction                      Unit price paid for the item on this order line.
           product_price                           Product Price     product                             Catalogue/list unit price of the product.
             price_delta Order Item Product Price, Product Price transaction                             order_item_product_price - product_price.
     order_item_discount                     Order Item Discount transaction                           Absolute discount applied on the item line.
order_item_discount_rate                Order Item Discount Rate transaction          

In [16]:
# Deterministic cap BEFORE any modelling (honest probabilities, fixed seed).
cap = cap_before_split(df.copy(), n_rows=DATACO_MAX_ROWS, seed=GLOBAL_SEED)
print(f"capped work frame: {len(cap):,} rows")
# NOTE: aggregates below are computed on `cap` only to illustrate the mechanics;
# the production pipeline derives them from TRAIN exclusively (see section 14).
raw_feats = build_raw_ml_features(cap, compute_order_level(cap))
print(raw_feats.shape)
raw_feats.head(5)

capped work frame: 40,000 rows


(40000, 17)


,order_item_quantity,order_item_product_price,product_price,order_item_discount,order_item_discount_rate,order_item_profit_ratio,order_item_total,days_schedule,price_delta,year,month,day,day_of_week,hour,is_weekend,item_count_per_order,order_total_value
0,1.0,199.990005,199.990005,24.0,0.12,0.06,175.990005,4.0,0.0,2016.0,4.0,1.0,4.0,21.0,0.0,2.0,294.280006
1,5.0,50.000000,50.000000,5.0,0.02,0.04,245.000000,1.0,0.0,2017.0,6.0,9.0,4.0,18.0,0.0,1.0,245.000000
2,5.0,49.980000,49.980000,5.0,0.02,0.48,244.899994,4.0,0.0,2015.0,1.0,2.0,4.0,14.0,0.0,1.0,244.899994
3,1.0,299.980011,299.980011,48.0,0.16,0.47,251.979996,4.0,0.0,2017.0,1.0,10.0,1.0,1.0,0.0,1.0,251.979996
4,3.0,39.990002,39.990002,12.0,0.10,-0.20,107.970001,4.0,0.0,2017.0,4.0,6.0,3.0,7.0,0.0,1.0,107.970001


In [17]:
print("one-hot categorical sources:", categorical_source_columns())
print("(cardinality <= 15 are one-hot encoded; higher-cardinality ones stay metadata-only)")

one-hot categorical sources: ['Type', 'Market', 'Shipping Mode', 'Customer Segment', 'Department Name']
(cardinality <= 15 are one-hot encoded; higher-cardinality ones stay metadata-only)


## 14. Train / validation / test split (leakage-safe)

In [18]:
from src.preprocessing.leakage import split_dataset
split_result = split_dataset(
    cap, ratios=SPLIT_RATIOS, strategy=SPLIT_STRATEGY, seed=GLOBAL_SEED,
)
print("sizes:", split_result.report["sizes"])
print("identity overlap checks:")
print(json.dumps(split_result.report["identity_overlap_checks"], indent=1))

sizes: {'train': 28000, 'validation': 6000, 'test': 6000}
identity overlap checks:
{
 "Order Id": {
  "train_vs_validation": 0,
  "train_vs_test": 0,
  "validation_vs_test": 0
 },
 "Order Customer Id": {
  "train_vs_validation": 2910,
  "train_vs_test": 2927,
  "validation_vs_test": 1103
 },
 "Product Card Id": {
  "train_vs_validation": 109,
  "train_vs_test": 109,
  "validation_vs_test": 104
 }
}


## 15. Save processed datasets (fit on TRAIN only)

In [19]:
from src.preprocessing.dataco_pipeline import DataCoPreprocessor, PreprocessorConfig
from src.pipeline_v02 import save_processed_splits
preprocessor = DataCoPreprocessor(PreprocessorConfig(seed=GLOBAL_SEED, drop_empty_target=True))
processed = preprocessor.transform_all(
    split_result.train, split_result.validation, split_result.test,
)
print("preprocessor.fit_report:")
print(json.dumps(preprocessor.fit_report, indent=1)[:1600])
written = save_processed_splits(processed,
    {n: PROCESSED_DATA_DIR / n for n in ("train", "validation", "test")})
for p in written:
    print("wrote", p)

preprocessor.fit_report:
{
 "fitted_on_rows": 28000,
 "ml_feature_count": 43,
 "numeric_feature_count": 17,
 "onehot_column_count": 32,
 "ml_feature_columns": [
  "order_item_quantity",
  "order_item_product_price",
  "product_price",
  "order_item_discount",
  "order_item_discount_rate",
  "order_item_profit_ratio",
  "order_item_total",
  "days_schedule",
  "year",
  "month",
  "day",
  "day_of_week",
  "hour",
  "is_weekend",
  "item_count_per_order",
  "order_total_value",
  "Type_DEBIT",
  "Type_TRANSFER",
  "Type_PAYMENT",
  "Type_CASH",
  "Market_LATAM",
  "Market_Europe",
  "Market_Pacific Asia",
  "Market_USCA",
  "Market_Africa",
  "Shipping Mode_Standard Class",
  "Shipping Mode_Second Class",
  "Shipping Mode_First Class",
  "Shipping Mode_Same Day",
  "Customer Segment_Consumer",
  "Customer Segment_Corporate",
  "Customer Segment_Home Office",
  "Department Name_Fan Shop",
  "Department Name_Apparel",
  "Department Name_Golf",
  "Department Name_Footwear",
  "Department N

wrote train_metadata
wrote train_features
wrote train_target
wrote validation_metadata
wrote validation_features
wrote validation_target
wrote test_metadata
wrote test_features
wrote test_target


In [20]:
import hashlib
h = hashlib.md5((RAW_DATA_DIR / "DataCoSupplyChainDataset.csv").read_bytes()).hexdigest()
print("raw data untouched:", h == "354eb3cd362619ef120ca2201337ae92")

raw data untouched: True


## 16. Final summary

In [21]:
print("Reproducibility record (split)")
print(json.dumps(split_result.report, indent=1))
print()
print("feature columns per split:", processed["train"].ml_features.shape)
print()
print("CYBER-LABEL STATEMENT: the real DataCo dataset provides no native cybersecurity attack labels.")
print("V0.2 supplies leakage-safe ML features + versioned splits only (anomaly/attack modelling is deferred).")

Reproducibility record (split)
{
 "strategy": "order_grouped",
 "seed": 42,
 "ratios": {
  "train": 0.7,
  "validation": 0.15,
  "test": 0.15
 },
 "sizes": {
  "train": 28000,
  "validation": 6000,
  "test": 6000
 },
 "size_percentages": {
  "train": 0.7,
  "validation": 0.15,
  "test": 0.15
 },
 "identity_overlap_checks": {
  "Order Id": {
   "train_vs_validation": 0,
   "train_vs_test": 0,
   "validation_vs_test": 0
  },
  "Order Customer Id": {
   "train_vs_validation": 2910,
   "train_vs_test": 2927,
   "validation_vs_test": 1103
  },
  "Product Card Id": {
   "train_vs_validation": 109,
   "train_vs_test": 109,
   "validation_vs_test": 104
  }
 },
 "leakage_warning": false,
 "target_balance": {
  "train": {
   "total": 28000,
   "positive": 15354,
   "positive_rate": 0.548357
  },
  "validation": {
   "total": 6000,
   "positive": 3324,
   "positive_rate": 0.554
  },
  "test": {
   "total": 6000,
   "positive": 3223,
   "positive_rate": 0.537167
  }
 }
}

feature columns per split